# Spark-basiertes Multi-Algorithmus Hyperparameter-Tuning

## Verteilte Optimierung auf Hadoop/Spark Cluster

Dieses Notebook nutzt **Apache Spark** zur parallelen Hyperparameter-Optimierung für drei evolutionäre Algorithmen:

**Algorithmen:**
- **NSGA-II**: Non-dominated Sorting Genetic Algorithm II
- **SMS-EMOA**: S-Metric Selection Evolutionary Multi-Objective Algorithm
- **NSGA-III**: Reference-Point Based NSGA

**Vorteile der Cluster-Ausführung:**
- Parallele Trial-Ausführung auf mehreren Executors
- Höhere R (Simulationen) für stabilere Fitness-Bewertung
- Größere Populationen und mehr Generationen möglich
- Mehr Trials für feinere Hyperparameter-Suche

**Zielmetrik:** Hypervolume (maximieren)

---

## 1. Setup & Installation

In [1]:
# Pakete installieren (falls nicht vorhanden)
%pip install numpy pymoo pandas matplotlib seaborn --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import random
import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sys
from pathlib import Path
sys.path.insert(0, str(Path('.').resolve().parent))
import json
import warnings
warnings.filterwarnings('ignore')

# Spark
from pyspark.sql import SparkSession
from pyspark import SparkContext, SparkConf

# Pymoo
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.algorithms.moo.nsga3 import NSGA3
from pymoo.algorithms.moo.sms import SMSEMOA
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.operators.sampling.rnd import FloatRandomSampling
from pymoo.optimize import minimize
from pymoo.termination import get_termination
from pymoo.indicators.hv import HV
from pymoo.util.ref_dirs import get_reference_directions

# Matplotlib Style (kompatibel mit verschiedenen Versionen)
try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    try:
        plt.style.use('seaborn-whitegrid')
    except OSError:
        print("Seaborn style not available, using default")

plt.rcParams['figure.figsize'] = (14, 10)
plt.rcParams['font.size'] = 12

print("Imports successful")

Imports successful


---

## 2. Spark Session Konfiguration

In [3]:
# ============================================================
# SPARK SESSION KONFIGURATION
# ============================================================
# Wähle den Modus:
#   - "yarn"  : Für Cluster-Ausführung (empfohlen)
#   - "local" : Für lokale Tests

SPARK_MODE = "yarn"  # ← ÄNDERN: "yarn" für Cluster, "local" für lokal

# Cluster-Konfiguration (ANPASSEN an dein Cluster!)
NUM_EXECUTORS = 10      # Anzahl der Worker
EXECUTOR_MEMORY = "4g"  # RAM pro Worker
EXECUTOR_CORES = 2      # Cores pro Worker
DRIVER_MEMORY = "4g"    # RAM für Driver

# Spark Session erstellen
builder = SparkSession.builder \
    .appName("TopTrumps_HyperparameterTuning")

if SPARK_MODE == "yarn":
    # YARN-Modus: Echtes Cluster
    spark = builder \
        .master("yarn") \
        .config("spark.submit.deployMode", "client") \
        .config("spark.executor.instances", str(NUM_EXECUTORS)) \
        .config("spark.executor.memory", EXECUTOR_MEMORY) \
        .config("spark.executor.cores", str(EXECUTOR_CORES)) \
        .config("spark.driver.memory", DRIVER_MEMORY) \
        .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
        .config("spark.yarn.appMasterEnv.PYSPARK_PYTHON", "/usr/bin/python3") \
        .getOrCreate()
else:
    # Lokaler Modus: Für Tests
    spark = builder \
        .master("local[*]") \
        .config("spark.driver.memory", DRIVER_MEMORY) \
        .getOrCreate()

sc = spark.sparkContext

# Cluster-Info anzeigen
print("=" * 50)
print("SPARK SESSION INFO")
print("=" * 50)
print(f"Mode: {SPARK_MODE}")
print(f"Spark Version: {spark.version}")
print(f"Application ID: {sc.applicationId}")
print(f"Master: {sc.master}")
print(f"Default Parallelism: {sc.defaultParallelism}")

if SPARK_MODE == "yarn":
    print(f"Requested Executors: {NUM_EXECUTORS}")
    print(f"Executor Memory: {EXECUTOR_MEMORY}")
    print(f"Executor Cores: {EXECUTOR_CORES}")
    
print("=" * 50)

Spark Version: 3.0.1
Application ID: local-1769777471690
Master: local[*]
Default Parallelism: 2


### 2.1 HDFS: Simulation auf Worker verteilen

**WICHTIG:** Vor der ersten Ausführung muss `simulation.py` auf HDFS hochgeladen werden!

In [4]:
# ============================================================
# HDFS: simulation.py auf alle Worker verteilen
# ============================================================
# ANPASSEN: Pfad zu simulation.py auf HDFS

HDFS_SIMULATION_PATH = "hdfs:///user/tim.strauss/libs/simulation.py"

# Datei zum Spark-Context hinzufügen (wird auf alle Executors verteilt)
sc.addPyFile(HDFS_SIMULATION_PATH)

print(f"✓ Added simulation.py from HDFS: {HDFS_SIMULATION_PATH}")
print("  This file is now available on all Spark executors.")

# ============================================================
# HDFS Upload-Anleitung (einmalig auf dem Cluster ausführen):
# ============================================================
# 
# 1. Verzeichnis erstellen:
#    hdfs dfs -mkdir -p /user/tim.strauss/libs/
#
# 2. Datei hochladen:
#    hdfs dfs -put simulation.py /user/tim.strauss/libs/
#
# 3. Überprüfen:
#    hdfs dfs -ls /user/tim.strauss/libs/
# ============================================================

✓ Added simulation.py from HDFS: hdfs:///user/tim.strauss/libs/simulation.py
  This file is now available on all Spark executors.


---

## 3. Konfiguration

In [5]:
# Verzeichnisse
RESULTS_DIR = Path("../results")
PLOTS_DIR = Path("../plots")
RESULTS_DIR.mkdir(exist_ok=True)
PLOTS_DIR.mkdir(exist_ok=True)

# Problem-Parameter (aus config.json falls vorhanden)
try:
    with open(RESULTS_DIR / 'config.json', 'r') as f:
        CONFIG = json.load(f)
    SEED = CONFIG['seed']
    K = CONFIG['K']
    L = CONFIG['L']
    XL = CONFIG['xl']
    XU = CONFIG['xu']
    print(f"Config loaded: K={K}, L={L}, Seed={SEED}")
except FileNotFoundError:
    print("config.json not found, using defaults")
    SEED = 42
    K = 22
    L = 4
    XL = 1.0
    XU = 10.0

def set_seeds(seed):
    np.random.seed(seed)
    random.seed(seed)

config.json not found, using defaults


In [6]:
# ============================================================
# CLUSTER-OPTIMIERTE PARAMETER
# ============================================================

# Tuning-Parameter (erweitert für Cluster)
N_TRIALS = 50                # Trials pro Algorithmus (erhöht von 30)
N_GEN = 100                  # Generationen pro Trial (erhöht von 40)
R_SIMULATIONS = 1000         # Simulationen pro Evaluation (erhöht von 500)
TUNING_SEEDS = [42, 101, 202, 303, 404]  # 5 Seeds für robuste Bewertung

# Algorithmen zum Optimieren
ALGORITHMS = ['NSGA-II', 'SMS-EMOA', 'NSGA-III']

# Hyperparameter-Suchräume (erweitert)
SEARCH_SPACE = {
    'pop_size': (50, 200, 10),        # (min, max, step)
    'eta_crossover': (5.0, 30.0),     # (min, max)
    'eta_mutation': (5.0, 30.0),      # (min, max)
    'crossover_prob': (0.7, 1.0),     # (min, max)
    'n_partitions': (5, 15),          # Für NSGA-III
}

print("=" * 60)
print("CLUSTER HYPERPARAMETER TUNING CONFIGURATION")
print("=" * 60)
print(f"Algorithms: {ALGORITHMS}")
print(f"Trials per Algorithm: {N_TRIALS}")
print(f"Generations per Trial: {N_GEN}")
print(f"Simulations per Evaluation: {R_SIMULATIONS}")
print(f"Evaluation Seeds: {TUNING_SEEDS}")
print(f"Total Trials: {len(ALGORITHMS) * N_TRIALS}")
print(f"\nSearch Space:")
for param, bounds in SEARCH_SPACE.items():
    print(f"  {param}: {bounds}")

CLUSTER HYPERPARAMETER TUNING CONFIGURATION
Algorithms: ['NSGA-II', 'SMS-EMOA', 'NSGA-III']
Trials per Algorithm: 50
Generations per Trial: 100
Simulations per Evaluation: 1000
Evaluation Seeds: [42, 101, 202, 303, 404]
Total Trials: 150

Search Space:
  pop_size: (50, 200, 10)
  eta_crossover: (5.0, 30.0)
  eta_mutation: (5.0, 30.0)
  crossover_prob: (0.7, 1.0)
  n_partitions: (5, 15)


---

## 4. Simulation Setup

In [7]:
from simulation import TopTrumpsSimulation, TopTrumpsBalancing

# Lokale Simulation für Tests
sim = TopTrumpsSimulation(num_cards=K, num_categories=L)
print(f"Simulation created: {K} cards, {L} categories")

Simulation created: 22 cards, 4 categories


---

## 5. Trial-Parameter Generator

In [8]:
def generate_random_params(algorithm, seed):
    """
    Generiert zufällige Hyperparameter für einen Trial.
    
    Returns:
        dict: Parameter-Dictionary für den Trial
    """
    np.random.seed(seed)
    
    pop_min, pop_max, pop_step = SEARCH_SPACE['pop_size']
    eta_c_min, eta_c_max = SEARCH_SPACE['eta_crossover']
    eta_m_min, eta_m_max = SEARCH_SPACE['eta_mutation']
    prob_min, prob_max = SEARCH_SPACE['crossover_prob']
    
    params = {
        'algorithm': algorithm,
        'pop_size': int(np.random.choice(range(pop_min, pop_max + 1, pop_step))),
        'eta_crossover': np.random.uniform(eta_c_min, eta_c_max),
        'eta_mutation': np.random.uniform(eta_m_min, eta_m_max),
        'crossover_prob': np.random.uniform(prob_min, prob_max),
        'trial_seed': seed,
    }
    
    # NSGA-III braucht n_partitions
    if algorithm == 'NSGA-III':
        part_min, part_max = SEARCH_SPACE['n_partitions']
        params['n_partitions'] = int(np.random.randint(part_min, part_max + 1))
    
    return params


# Alle Trial-Konfigurationen generieren
all_trial_configs = []
trial_id = 0

for algo in ALGORITHMS:
    for i in range(N_TRIALS):
        config = generate_random_params(algo, seed=SEED + trial_id)
        config['trial_id'] = trial_id
        all_trial_configs.append(config)
        trial_id += 1

print(f"Generated {len(all_trial_configs)} trial configurations")
print(f"\nSample config:")
print(json.dumps(all_trial_configs[0], indent=2))

Generated 150 trial configurations

Sample config:
{
  "algorithm": "NSGA-II",
  "pop_size": 110,
  "eta_crossover": 24.91357467150582,
  "eta_mutation": 9.585869746654094,
  "crossover_prob": 0.9339073000818308,
  "trial_seed": 42,
  "trial_id": 0
}


---

## 6. Spark-kompatible Objective Function

In [9]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('.').resolve().parent))

def run_single_trial(params_dict):
    """
    Führt einen einzelnen Hyperparameter-Trial auf einem Spark-Executor aus.
    
    WICHTIG: Alle Imports müssen INNERHALB der Funktion sein,
    da diese auf den Executors ausgeführt wird.
    
    Args:
        params_dict: Dictionary mit Hyperparametern und Konfiguration
        
    Returns:
        dict: Trial-Ergebnis mit Hypervolume und Statistiken
    """
    # === LOKALE IMPORTS (wichtig für Spark!) ===
    import numpy as np
    import random
    import time
    
    from pymoo.algorithms.moo.nsga2 import NSGA2
    from pymoo.algorithms.moo.nsga3 import NSGA3
    from pymoo.algorithms.moo.sms import SMSEMOA
    from pymoo.operators.crossover.sbx import SBX
    from pymoo.operators.mutation.pm import PM
    from pymoo.operators.sampling.rnd import FloatRandomSampling
    from pymoo.optimize import minimize
    from pymoo.termination import get_termination
    from pymoo.indicators.hv import HV
    from pymoo.util.ref_dirs import get_reference_directions
    
    # Simulation importieren (muss auf allen Workern verfügbar sein!)
    from simulation import TopTrumpsSimulation, TopTrumpsBalancing
    
    # === PARAMETER EXTRAHIEREN ===
    algorithm_name = params_dict['algorithm']
    pop_size = params_dict['pop_size']
    eta_crossover = params_dict['eta_crossover']
    eta_mutation = params_dict['eta_mutation']
    crossover_prob = params_dict['crossover_prob']
    trial_seed = params_dict['trial_seed']
    trial_id = params_dict['trial_id']
    
    # Globale Konfiguration (muss mitgeliefert werden)
    K = params_dict.get('K', 22)
    L = params_dict.get('L', 4)
    XL = params_dict.get('XL', 1.0)
    XU = params_dict.get('XU', 10.0)
    R_SIMS = params_dict.get('R_SIMULATIONS', 1000)
    N_GEN = params_dict.get('N_GEN', 100)
    EVAL_SEEDS = params_dict.get('EVAL_SEEDS', [42, 101, 202, 303, 404])
    
    # === SIMULATION ERSTELLEN ===
    sim = TopTrumpsSimulation(num_cards=K, num_categories=L)
    problem = TopTrumpsBalancing(sim, n_simulations=R_SIMS, xl=XL, xu=XU)
    termination = get_termination("n_gen", N_GEN)
    
    # === ALGORITHMUS ERSTELLEN ===
    def create_algorithm():
        if algorithm_name == 'NSGA-II':
            return NSGA2(
                pop_size=pop_size,
                sampling=FloatRandomSampling(),
                crossover=SBX(prob=crossover_prob, eta=eta_crossover),
                mutation=PM(eta=eta_mutation),
                eliminate_duplicates=True
            )
        elif algorithm_name == 'SMS-EMOA':
            return SMSEMOA(
                pop_size=pop_size,
                sampling=FloatRandomSampling(),
                crossover=SBX(prob=crossover_prob, eta=eta_crossover),
                mutation=PM(eta=eta_mutation),
                eliminate_duplicates=True
            )
        elif algorithm_name == 'NSGA-III':
            n_partitions = params_dict.get('n_partitions', 12)
            ref_dirs = get_reference_directions("das-dennis", 2, n_partitions=n_partitions)
            return NSGA3(
                pop_size=pop_size,
                ref_dirs=ref_dirs,
                sampling=FloatRandomSampling(),
                crossover=SBX(prob=crossover_prob, eta=eta_crossover),
                mutation=PM(eta=eta_mutation),
                eliminate_duplicates=True
            )
        else:
            raise ValueError(f"Unknown algorithm: {algorithm_name}")
    
    # === EVALUATION ÜBER MEHRERE SEEDS ===
    hvs = []
    n_solutions_list = []
    start_time = time.time()
    
    for eval_seed in EVAL_SEEDS:
        np.random.seed(eval_seed)
        random.seed(eval_seed)
        
        algorithm = create_algorithm()
        
        try:
            res = minimize(problem, algorithm, termination, seed=eval_seed, verbose=False)
            hv = HV(ref_point=np.array([0.0, 0.0]))(res.F)
            hvs.append(hv)
            n_solutions_list.append(len(res.F))
        except Exception as e:
            # Bei Fehler: NaN zurückgeben
            hvs.append(float('nan'))
            n_solutions_list.append(0)
    
    total_time = time.time() - start_time
    
    # === ERGEBNIS ZUSAMMENSTELLEN ===
    result = {
        'trial_id': trial_id,
        'algorithm': algorithm_name,
        'pop_size': pop_size,
        'eta_crossover': eta_crossover,
        'eta_mutation': eta_mutation,
        'crossover_prob': crossover_prob,
        'hypervolume_mean': float(np.nanmean(hvs)),
        'hypervolume_std': float(np.nanstd(hvs)),
        'hypervolume_min': float(np.nanmin(hvs)),
        'hypervolume_max': float(np.nanmax(hvs)),
        'n_solutions_mean': float(np.mean(n_solutions_list)),
        'runtime_seconds': total_time,
        'status': 'completed' if not np.isnan(np.nanmean(hvs)) else 'failed'
    }
    
    if algorithm_name == 'NSGA-III':
        result['n_partitions'] = params_dict.get('n_partitions')
    
    return result


print("Objective function defined.")

Objective function defined.


---

## 7. Lokaler Test (Optional)

In [10]:
# Schneller lokaler Test mit einem Trial
TEST_LOCAL = False  # Auf False setzen um zu überspringen

if TEST_LOCAL:
    print("Running local test with reduced parameters...")
    
    # Test-Config mit reduzierten Parametern
    test_config = {
        'trial_id': -1,
        'algorithm': 'NSGA-II',
        'pop_size': 50,
        'eta_crossover': 15.0,
        'eta_mutation': 20.0,
        'crossover_prob': 0.9,
        'trial_seed': 42,
        'K': K,
        'L': L,
        'XL': XL,
        'XU': XU,
        'R_SIMULATIONS': 100,  # Reduziert für Test
        'N_GEN': 20,           # Reduziert für Test
        'EVAL_SEEDS': [42],    # Nur 1 Seed für Test
    }
    
    test_result = run_single_trial(test_config)
    print(f"\nTest Result:")
    print(json.dumps(test_result, indent=2))
    print("\nLocal test passed!")

---

## 8. Spark-Parallelisierung

In [11]:
# Globale Konfiguration zu jedem Trial hinzufügen
for config in all_trial_configs:
    config['K'] = K
    config['L'] = L
    config['XL'] = XL
    config['XU'] = XU
    config['R_SIMULATIONS'] = R_SIMULATIONS
    config['N_GEN'] = N_GEN
    config['EVAL_SEEDS'] = TUNING_SEEDS

print(f"Prepared {len(all_trial_configs)} trial configurations with global params")

Prepared 150 trial configurations with global params


In [12]:
# ============================================================
# SPARK-AUSFÜHRUNG
# ============================================================

print("=" * 60)
print("STARTING SPARK-BASED HYPERPARAMETER TUNING")
print("=" * 60)
print(f"Total Trials: {len(all_trial_configs)}")
print(f"Spark Parallelism: {sc.defaultParallelism}")
print("\nThis may take a while...")
print("=" * 60)

start_time = time.time()

# Trial-Konfigurationen als RDD verteilen
trials_rdd = sc.parallelize(all_trial_configs, numSlices=min(len(all_trial_configs), sc.defaultParallelism * 2))

# Trials parallel ausführen
results_rdd = trials_rdd.map(run_single_trial)

# Ergebnisse sammeln
all_results = results_rdd.collect()

total_time = time.time() - start_time

print(f"\n{'=' * 60}")
print("TUNING COMPLETED!")
print(f"{'=' * 60}")
print(f"Total Runtime: {total_time:.1f} seconds ({total_time/60:.1f} minutes)")
print(f"Completed Trials: {len([r for r in all_results if r['status'] == 'completed'])}")
print(f"Failed Trials: {len([r for r in all_results if r['status'] == 'failed'])}")

STARTING SPARK-BASED HYPERPARAMETER TUNING
Total Trials: 150
Spark Parallelism: 2

This may take a while...


[Stage 0:>                                                          (0 + 2) / 4]
Compiled modules for significant speedup can not be used!
https://pymoo.org/installation.html#installation

To disable this warning:
from pymoo.config import Config
Config.warnings['not_compiled'] = False


Compiled modules for significant speedup can not be used!
https://pymoo.org/installation.html#installation

To disable this warning:
from pymoo.config import Config
Config.warnings['not_compiled'] = False



KeyboardInterrupt: 

---

## 9. Ergebnis-Analyse

In [ ]:
# DataFrame erstellen
results_df = pd.DataFrame(all_results)

# Nur erfolgreiche Trials
completed_df = results_df[results_df['status'] == 'completed'].copy()

print(f"Completed Trials: {len(completed_df)}")
print(f"\nTrials per Algorithm:")
print(completed_df.groupby('algorithm').size())

# Top 5 pro Algorithmus
print("\n" + "=" * 60)
print("TOP 5 CONFIGURATIONS PER ALGORITHM")
print("=" * 60)

for algo in ALGORITHMS:
    algo_df = completed_df[completed_df['algorithm'] == algo].sort_values('hypervolume_mean', ascending=False)
    print(f"\n{algo}:")
    print(algo_df[['pop_size', 'eta_crossover', 'eta_mutation', 'crossover_prob', 'hypervolume_mean', 'hypervolume_std']].head(5).to_string(index=False))

In [ ]:
# Beste Konfiguration pro Algorithmus
best_configs = {}

print("=" * 60)
print("BEST CONFIGURATION PER ALGORITHM")
print("=" * 60)

for algo in ALGORITHMS:
    algo_df = completed_df[completed_df['algorithm'] == algo]
    if len(algo_df) > 0:
        best_idx = algo_df['hypervolume_mean'].idxmax()
        best = algo_df.loc[best_idx].to_dict()
        best_configs[algo] = best
        
        print(f"\n{algo}:")
        print(f"  Hypervolume: {best['hypervolume_mean']:.4f} ± {best['hypervolume_std']:.4f}")
        print(f"  pop_size: {best['pop_size']}")
        print(f"  eta_crossover: {best['eta_crossover']:.2f}")
        print(f"  eta_mutation: {best['eta_mutation']:.2f}")
        print(f"  crossover_prob: {best['crossover_prob']:.2f}")
        if algo == 'NSGA-III' and 'n_partitions' in best:
            print(f"  n_partitions: {best['n_partitions']}")

In [ ]:
# Algorithmus-Vergleich
print("\n" + "=" * 60)
print("ALGORITHM COMPARISON (Best Hypervolume)")
print("=" * 60)

comparison = []
for algo, config in best_configs.items():
    comparison.append({
        'Algorithm': algo,
        'Best HV': config['hypervolume_mean'],
        'Std': config['hypervolume_std'],
        'pop_size': config['pop_size'],
        'Runtime (s)': config['runtime_seconds']
    })

comparison_df = pd.DataFrame(comparison).sort_values('Best HV', ascending=False)
print(comparison_df.to_string(index=False))

# Bester Algorithmus
best_algo = comparison_df.iloc[0]['Algorithm']
print(f"\nBest Algorithm: {best_algo} (HV = {comparison_df.iloc[0]['Best HV']:.4f})")

---

## 10. Visualisierung

In [ ]:
# Hypervolume-Verteilung pro Algorithmus
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

colors = {'NSGA-II': '#2E86AB', 'SMS-EMOA': '#E94F37', 'NSGA-III': '#3DDC97'}

for i, algo in enumerate(ALGORITHMS):
    algo_df = completed_df[completed_df['algorithm'] == algo]
    axes[i].hist(algo_df['hypervolume_mean'], bins=15, color=colors[algo], edgecolor='white', alpha=0.8)
    axes[i].axvline(best_configs[algo]['hypervolume_mean'], color='red', linestyle='--', linewidth=2, label=f'Best: {best_configs[algo]["hypervolume_mean"]:.3f}')
    axes[i].set_xlabel('Hypervolume')
    axes[i].set_ylabel('Count')
    axes[i].set_title(f'{algo}')
    axes[i].legend()

plt.suptitle('Hypervolume Distribution per Algorithm', fontsize=14)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'spark_tuning_hv_distribution.svg', format='svg', bbox_inches='tight')
plt.show()

In [ ]:
# Boxplot-Vergleich
fig, ax = plt.subplots(figsize=(10, 6))

data_for_plot = [completed_df[completed_df['algorithm'] == algo]['hypervolume_mean'].values for algo in ALGORITHMS]

bp = ax.boxplot(data_for_plot, labels=ALGORITHMS, patch_artist=True,
                flierprops={'marker': 'x', 'markersize': 5, 'markeredgecolor': 'black', 'markeredgewidth': 1})

for patch, algo in zip(bp['boxes'], ALGORITHMS):
    patch.set_facecolor(colors[algo])
    patch.set_alpha(0.7)

ax.set_ylabel('Hypervolume')
ax.set_title('Algorithm Comparison: Hypervolume Distribution')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'spark_tuning_comparison.svg', format='svg', bbox_inches='tight')
plt.show()

In [ ]:
# Parameter Importance: pop_size vs Hypervolume
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for i, algo in enumerate(ALGORITHMS):
    algo_df = completed_df[completed_df['algorithm'] == algo]
    axes[i].scatter(algo_df['pop_size'], algo_df['hypervolume_mean'], 
                   c=colors[algo], alpha=0.6, s=50)
    
    # Trendlinie
    z = np.polyfit(algo_df['pop_size'], algo_df['hypervolume_mean'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(algo_df['pop_size'].min(), algo_df['pop_size'].max(), 100)
    axes[i].plot(x_line, p(x_line), 'r--', alpha=0.8, linewidth=2)
    
    axes[i].set_xlabel('Population Size')
    axes[i].set_ylabel('Hypervolume')
    axes[i].set_title(f'{algo}')

plt.suptitle('Impact of Population Size on Hypervolume', fontsize=14)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'spark_tuning_popsize_impact.svg', format='svg', bbox_inches='tight')
plt.show()

---

## 11. Ergebnisse speichern

In [ ]:
# Alle Ergebnisse speichern
results_export = {
    'config': {
        'K': K,
        'L': L,
        'N_TRIALS': N_TRIALS,
        'N_GEN': N_GEN,
        'R_SIMULATIONS': R_SIMULATIONS,
        'ALGORITHMS': ALGORITHMS,
        'SEARCH_SPACE': {k: list(v) if isinstance(v, tuple) else v for k, v in SEARCH_SPACE.items()},
        'TUNING_SEEDS': TUNING_SEEDS,
        'total_runtime_seconds': total_time
    },
    'all_trials': all_results,
    'best_per_algorithm': {algo: {k: (float(v) if isinstance(v, (np.floating, np.integer)) else v) 
                                   for k, v in config.items()} 
                          for algo, config in best_configs.items()}
}

with open(RESULTS_DIR / 'spark_tuning_results.json', 'w') as f:
    json.dump(results_export, f, indent=2, default=str)

print(f"Results saved to: {RESULTS_DIR / 'spark_tuning_results.json'}")

In [ ]:
# Beste Hyperparameter als separate Datei
best_hyperparameters = {}

for algo, config in best_configs.items():
    best_hyperparameters[algo] = {
        'pop_size': int(config['pop_size']),
        'eta_crossover': float(config['eta_crossover']),
        'eta_mutation': float(config['eta_mutation']),
        'crossover_prob': float(config['crossover_prob']),
        'hypervolume': float(config['hypervolume_mean']),
        'hypervolume_std': float(config['hypervolume_std'])
    }
    if algo == 'NSGA-III' and 'n_partitions' in config:
        best_hyperparameters[algo]['n_partitions'] = int(config['n_partitions'])

with open(RESULTS_DIR / 'best_hyperparameters.json', 'w') as f:
    json.dump(best_hyperparameters, f, indent=2)

print(f"Best hyperparameters saved to: {RESULTS_DIR / 'best_hyperparameters.json'}")
print("\nContent:")
print(json.dumps(best_hyperparameters, indent=2))

---

## 12. Spark Session beenden

In [ ]:
# Spark Session stoppen
spark.stop()
print("Spark session stopped.")

---

## 13. Zusammenfassung & Verwendung

### Ergebnisse

Die optimierten Hyperparameter wurden gespeichert in:
- `results/spark_tuning_results.json` - Alle Trial-Ergebnisse
- `results/best_hyperparameters.json` - Beste Konfiguration pro Algorithmus

### Verwendung in anderen Notebooks

```python
# Optimierte Parameter laden
with open('../results/best_hyperparameters.json', 'r') as f:
    best_params = json.load(f)

# Beispiel: NSGA-II mit optimierten Parametern
params = best_params['NSGA-II']
algorithm = NSGA2(
    pop_size=params['pop_size'],
    crossover=SBX(prob=params['crossover_prob'], eta=params['eta_crossover']),
    mutation=PM(eta=params['eta_mutation'])
)
```

### Cluster-Ausführung

Für die Ausführung auf dem Hadoop-Cluster:

1. Notebook als Python-Script exportieren oder `spark-submit` nutzen
2. `simulation.py` muss auf allen Worker-Nodes verfügbar sein
3. Spark-Konfiguration an Cluster anpassen (Memory, Cores)